In [3]:
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph.message import add_messages

class FormsState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    qa_answer: str
    user_input: str
    # filled_pdf_path: str

state: FormsState = {
    "messages":  [HumanMessage(content="điền cho tôi mẫu CC01")],
    "qa_answer": "Để cấp căn cước lần đầu bạn cần 2 mẫu: CC01 tại https://s3.amazonaws.com/MauCC01.pdf và CC02 tại https://s3.amazonaws.com/MauCC02.pdf",
    "user_input": "điền cho tôi mẫu CC01",
}

In [4]:
from langchain_core.tools import tool

@tool
def select_forms(qa_answer: str, user_input: int):
    """
    Phân tích câu trả lời từ QA node.
    """

    if user_input > 1:
        return {
            "messages" : f"Đã phân tích nhiều câu trả lời {qa_answer}",
            "value": 2
        }
    else:
        return {
            "messages" : f"Đã phân tích một câu trả lời {qa_answer}",
            "value": 1
        }
        

@tool
def result(phantich: dict):
    """
    Đưa ra kết luận từ  kết quả phân tích

    Args:
        phantich (dict): _description_
    """
    if phantich["value"] == 2:
        return phantich["messages"]
    else:
        return f"Câu trả lời duy nhất {phantich["messages"]}"

from langchain_groq import ChatGroq
from langchain.messages import HumanMessage, SystemMessage, AnyMessage

GROQ_API_KEY = "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"

_llm = ChatGroq(
    api_key=GROQ_API_KEY,
    model="openai/gpt-oss-120b",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=5,
)

llm_with_tool = _llm.bind_tools([select_forms, result])

d:\capstone-project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
SYSTEM = "Bạn là agent phân tích. Dùng tool để phân tích và đưa ra kết quả"

user = {
    "qa_answer": "Để cấp căn cước lần đầu bạn cần 2 mẫu: CC01 tại https://s3.amazonaws.com/MauCC01.pdf và CC02 tại https://s3.amazonaws.com/MauCC02.pdf",
    "user_input": 3,
}

user_content = f"""
qa_answer: {user["qa_answer"]}
user_input: {user["user_input"]}
"""

messages = [SystemMessage(content=SYSTEM)]
messages.append(HumanMessage(content=user_content))

TOOL_MAP = {"select_forms": select_forms, "result": result}

# Lặp cho đến khi LLM không gọi tool nữa
while True:
    response = llm_with_tool.invoke(messages)
    print(f"LOG[RESPONE]: {response}")
    messages.append(response)

    if not response.tool_calls:
        print("\n✅ Kết quả cuối:")
        print(response.content)
        break

    for i, tc in enumerate(response.tool_calls):
        print(f"🔧 Bước {i+1}: Gọi tool [{tc['name']}] với args: {tc['args']}")
        
        tool_fn = TOOL_MAP[tc["name"]]
        tool_result = tool_fn.invoke(tc["args"])
        
        print(f"   ↳ Kết quả: {tool_result}")
        
        messages.append({
            "role": "tool",
            "tool_call_id": tc["id"],
            "content": str(tool_result)
        })

LOG[RESPONE]: content='' additional_kwargs={'reasoning_content': 'The user gave a QA answer and a user_input number. We need to use tool select_forms to analyze the answer with user_input. Then use result to produce conclusion. So call select_forms with qa_answer and user_input.', 'tool_calls': [{'id': 'fc_d732642d-a4d2-49c5-ab9e-a80208b13bd5', 'function': {'arguments': '{"qa_answer":"Để cấp căn cước lần đầu bạn cần 2 mẫu: CC01 tại https://s3.amazonaws.com/MauCC01.pdf và CC02 tại https://s3.amazonaws.com/MauCC02.pdf","user_input":3}', 'name': 'select_forms'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 124, 'prompt_tokens': 255, 'total_tokens': 379, 'completion_time': 0.27088571, 'completion_tokens_details': {'reasoning_tokens': 46}, 'prompt_time': 0.012766571, 'prompt_tokens_details': None, 'queue_time': 0.052628017, 'total_time': 0.283652281}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_45f51928b5', 'service_tier': 'on_demand', 'fi

In [ ]:
# messages.append(response)

# # Thực thi tool calls
# for tc in response.tool_calls:
#     tool_fn = {"select_forms": select_forms, "result": result}[tc["name"]]
#     tool_result = tool_fn.invoke(tc["args"])
#     messages.append({"role": "tool", "tool_call_id": tc["id"], "content": str(tool_result)})

# # Gọi LLM lần 2 để tổng hợp kết quả
# final = llm_with_tool.invoke(messages)
# print(final.content)
